# 🧰 Extra Topic: Prompt Engineering for Data Engineers

In Topics 1 to 9 we built a customer-facing chatbot. The chatbot is one shape of LLM application. Today I want to show you a completely different shape that uses the exact same primitives: a **batch enrichment pipeline**.

## Who this notebook is for

You run a daily ETL job at Barclays. The job lands a table of customer support tickets in the warehouse. Product wants every row to carry six new columns:

1. `intent` - one of `account_inquiry`, `card_services`, `loans`, `investments`, `complaint`, `general`.
2. `urgency` - `low`, `medium`, `high`.
3. `product_mentioned` - the Barclays product name extracted from the body.
4. `body_redacted` - the body with UK PII (NI numbers, sort codes, account numbers, postcodes) replaced by placeholders.
5. `suggested_response` - a draft reply grounded in the Barclays product PDFs.
6. `output_safe` - a boolean from the FCA financial-advice boundary check.

You have 50 rows today, 5,000 next month, 50,000 by Q3. You need a pipeline that is cheap, idempotent, observable, and never logs raw PII.

## What I am NOT going to do

I am not going to introduce new concepts. Every primitive in this notebook is a direct re-use of:

- Structured Outputs (`response_format={"type": "json_schema"}`) from Topic 4
- Embeddings + ChromaDB retrieval from Topic 6
- The PII regex registry and output validator from Topic 8
- Tenacity retry and per-request cost tracking from Topic 9

What is new is the **shape**: row-level fan-out instead of a chat loop. The same primitives compose differently.

## The hard rule for this notebook

This notebook is fully self-contained. It does NOT import anything from prior notebooks via Python globals. The only handoff is via the filesystem: S3 holds the Barclays product PDFs, this notebook downloads them fresh, runs the Topic 2 extract-clean-chunk pipeline inline, builds the ChromaDB collection inline, and then runs the enrichment pipeline. If S3 is unreachable we fall back to a small inline corpus so the demo still runs end-to-end.

## Section 0: Environment setup

Same setup pattern as every other notebook in this course. Pin every library, never hardcode keys, always pin `numpy<2` because chromadb 1.x will break on numpy 2.x.

In [ ]:
# openai 2.32.0 has Structured Outputs and Batch API support.
# chromadb 1.5.8 is the current stable; configuration={"hnsw":{"space":"cosine"}} is the modern API.
# tenacity 9.1.4 is the current stable retry library (requires Python 3.10+).
# pymupdf 1.27.2.2 is the canonical PDF text extractor (same pin as Topic 2).
# tiktoken 0.9.0 is pinned because 0.12.0+ dropped manylinux2014 wheels which break older SageMaker AL2 images.
# requests is used to download the PDFs over HTTPS from the public S3 bucket.
# sagemaker==2.232.1 is the last classic SDK release where get_execution_role lives at the top level.
# boto3 is the AWS SDK - needed for SageMaker session.
# numpy<2 is mandatory: chromadb 1.x breaks on numpy 2.x.
!pip install -q "openai==2.32.0" "chromadb==1.5.8" "tenacity==9.1.4" "pymupdf==1.27.2.2" "tiktoken==0.9.0" "requests" "sagemaker==2.232.1" "boto3" "numpy<2"

In [ ]:
# Standard library imports first.
import os
import re
import io
import json
import time
import uuid
import getpass
import logging
import unicodedata
from io import BytesIO
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Callable

# Third-party libraries.
import requests
import pymupdf
import pandas as pd
import sagemaker
from sagemaker import get_execution_role
from openai import OpenAI, OpenAIError, RateLimitError, APIConnectionError, APITimeoutError
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
import chromadb

# Load the OpenAI key via getpass so it never lands in the notebook output.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# SageMaker session and execution role - same pattern Topics 2-9 use.
sess = sagemaker.Session()
role = get_execution_role()
client = OpenAI()

# Constants used throughout the notebook.
MODEL = "gpt-4o"
EMBED_MODEL = "text-embedding-3-small"
MODERATION_MODEL = "omni-moderation-latest"
S3_BUCKET = "barclays-prompt-eng-data"
S3_REGION = "us-east-2"

# gpt-4o pricing (USD per 1K tokens) verified against OpenAI pricing as of April 2026.
GPT4O_INPUT_PRICE_PER_1K = 0.0025
GPT4O_OUTPUT_PRICE_PER_1K = 0.01
EMBED_PRICE_PER_1K = 0.00002

# JSON-line logger so every row produces one structured log entry.
# force=True is required when running inside Jupyter so basicConfig actually applies.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)
log = logging.getLogger("pipeline")
log.info("Pipeline session ready. role=%s, bucket=%s", role, S3_BUCKET)

### Filesystem-only handoff

I rebuild the Barclays product corpus from scratch in this notebook. I do NOT import `chunks`, `collection`, or any other variable from a prior topic. The reason is operational: in production a daily ETL job runs in its own process; it cannot rely on a developer notebook holding live state. The pipeline must be reproducible from cold storage every single run. Same rule applies here: cold start, fetch from S3, rebuild.

In [ ]:
# Download the canonical Barclays PDFs over HTTPS - same pattern as Topic 2/6/7/9.
# Public bucket, no IAM credentials needed.
S3_BASE_URL = f"https://{S3_BUCKET}.s3.{S3_REGION}.amazonaws.com"
DOWNLOAD_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/pdf,*/*",
}

# Two PDFs - matches the standardized T6/T7/T9 corpus for consistency across the course.
PDF_KEYS = [
    "barclays_personal_loan_faq.pdf",
    "barclays_credit_card_tnc.pdf",
]

# Inline fallback corpus used only if S3 is unreachable.
INLINE_FALLBACK_DOCS = [
    "Barclaycard Platinum credit card. Representative APR 24.9 percent variable. No annual fee. 0 percent on purchases for 18 months from account opening.",
    "Barclays Personal Loan. Loans from GBP 1,000 to GBP 50,000. Representative APR 6.5 percent on amounts of GBP 7,500 to GBP 25,000. Early repayment is permitted.",
    "Barclays Premier Savings. Variable rate of 4.10 percent AER paid monthly. Minimum opening balance GBP 1. Withdrawals permitted at any time without penalty.",
]


def fetch_pdf_bytes(key: str) -> Optional[bytes]:
    """Download a PDF from the public S3 bucket. Returns bytes or None on failure."""
    url = f"{S3_BASE_URL}/{key}"
    try:
        resp = requests.get(url, headers=DOWNLOAD_HEADERS, timeout=30)
        resp.raise_for_status()
        if not resp.content:
            raise RuntimeError(f"Empty PDF body for {key}")
        return resp.content
    except Exception as exc:
        log.warning("S3 fetch failed for %s: %s", key, exc)
        return None


# Download each PDF and extract its text using PyMuPDF.
# We do NOT write to disk - BytesIO keeps the bytes in memory for PyMuPDF to consume.
raw_docs: List[str] = []
for key in PDF_KEYS:
    payload = fetch_pdf_bytes(key)
    if payload is None:
        continue
    doc = pymupdf.open(stream=BytesIO(payload), filetype="pdf")
    text = "\n\n".join(page.get_text("text") for page in doc)
    doc.close()
    raw_docs.append(text)
    log.info("Loaded %s (%d chars)", key, len(text))

# Loud fallback if every download failed.
if not raw_docs:
    log.warning("S3 unreachable; using inline fallback corpus instead.")
    raw_docs = list(INLINE_FALLBACK_DOCS)

log.info("Corpus size: %d documents", len(raw_docs))

In [ ]:
# Same cleaning pipeline as Topic 2 - regex repair of common PDF artifacts.
def clean_pdf_text(text: str) -> str:
    # NFKC normalisation defends against full-width digits and homoglyphs.
    text = unicodedata.normalize("NFKC", text)
    # Fix hyphenated line breaks: 're-\npayment' -> 'repayment'.
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    # Strip the 'Barclays Bank UK PLC Page N of N' boilerplate that repeats every page.
    text = re.sub(r"Barclays Bank UK PLC\s+Page\s+\d+\s+of\s+\d+[^\n]*\n", "", text)
    # Collapse multiple spaces or tabs to one.
    text = re.sub(r"[ \t]+", " ", text)
    # Normalize line endings, then collapse 3+ newlines to 2.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# Same chunker as Topic 2 - sentence-boundary aware, fixed window with overlap.
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> List[str]:
    if not text:
        return []
    out, start = [], 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            # Try to break on a sentence boundary if we are at least halfway through the window.
            boundary = max(
                text.rfind(". ", start, end),
                text.rfind(".\n", start, end),
                text.rfind("\n\n", start, end),
            )
            if boundary > start + chunk_size // 2:
                end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            out.append(chunk)
        start += chunk_size - overlap
    return out


# Apply clean + chunk to every downloaded document.
cleaned = [clean_pdf_text(d) for d in raw_docs]
all_chunks: List[str] = []
for doc in cleaned:
    all_chunks.extend(chunk_text(doc, chunk_size=800, overlap=100))

log.info("Built %d chunks across %d documents", len(all_chunks), len(cleaned))

In [ ]:
# Build the ChromaDB collection inline. We use a separate path/name so we never
# collide with the T6/T7 collection on disk.
chroma_client = chromadb.PersistentClient(path="./chroma_db_extra08")
collection = chroma_client.get_or_create_collection(
    name="barclays_products_extra08",
    configuration={"hnsw": {"space": "cosine"}},
)


def embed_texts(texts: List[str]) -> List[List[float]]:
    """Embed a batch of strings via text-embedding-3-small. Returns a list of float lists."""
    # The Embeddings API accepts a list of strings in a single request - much faster than one-at-a-time.
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]


# Populate the collection only on first run. Re-running the cell is idempotent.
if collection.count() == 0:
    embeddings = embed_texts(all_chunks)
    ids = [f"chunk_{i}" for i in range(len(all_chunks))]
    collection.add(ids=ids, documents=all_chunks, embeddings=embeddings)
    log.info("Embedded and stored %d chunks", len(all_chunks))
else:
    log.info("Reusing existing collection with %d chunks", collection.count())

## 🏗️ The pipeline we are building

I am going to generate a small synthetic DataFrame of 50 customer support tickets. In production this would be a daily landing table from your ticketing system. The pipeline reads a row, runs four enrichments, and writes the enriched row to a downstream table.

<!-- DIAGRAM: pipeline-overview -->
[View diagram](../../plans/extra_08_data_engineer_usecase/diagrams/pipeline-overview.mmd)

The diagram above shows the full flow: the raw ticket lands as one row of the input DataFrame, passes through PII redaction first (so nothing past this point sees raw PII), then classification, then RAG grounding, then output validation. Every LLM call is wrapped in a tenacity retry. Every row produces a single JSON-line log entry containing prompt and completion tokens, latency, and the enrichment status. The Barclays PDFs we just chunked are the knowledge source for the RAG step.

In [ ]:
# Synthetic input data - 10 templates, replicated to 50 rows.
# In production this would be a SELECT from your ticketing system landing table.
# Note that several templates intentionally include realistic-looking UK PII (NI numbers,
# sort codes, postcodes, account numbers) so we can exercise the redaction step downstream.
SYNTHETIC_TICKETS = [
    "Hi, my Barclaycard was declined yesterday at Tesco. My account is 12345678 and sort code 12-34-56. Can you help?",
    "What is the current APR on the Barclays Personal Loan for GBP 10,000?",
    "I want to close my Premier Savings account. My NI is AB123456C.",
    "My card was lost on holiday. Please freeze it immediately. Account 87654321.",
    "Can I overpay my personal loan without penalty?",
    "I think I have been scammed. Someone called pretending to be from Barclays.",
    "What is the interest rate on the Premier Savings account today?",
    "Customer SW1A 1AA reports a duplicate direct debit on account 11223344.",
    "Please send me a statement for the last 6 months for sort code 20-00-00 account 99887766.",
    "I would like to apply for a personal loan. What APR can I get?",
]

# Build a 50-row DataFrame by recycling the templates and stamping each row with a UUID.
rows = []
for i in range(50):
    body = SYNTHETIC_TICKETS[i % len(SYNTHETIC_TICKETS)]
    rows.append({
        "ticket_id": str(uuid.uuid4()),
        "received_at": pd.Timestamp.utcnow().isoformat(),
        "body": body,
    })

tickets_df = pd.DataFrame(rows)
log.info("Synthetic input: %d rows", len(tickets_df))
tickets_df.head(3)

## 🎯 Enrichment 1: structured classification with Structured Outputs

For a chatbot you can tolerate a free-form response. For a data pipeline you cannot. Every row must produce a typed dict that lands cleanly in DataFrame columns. The right OpenAI primitive here is `response_format={"type": "json_schema", "strict": true}` (Structured Outputs). It guarantees the model returns valid JSON conforming to the schema OR refuses; it never produces malformed text.

I am asking for three fields per ticket: `intent`, `urgency`, `product_mentioned`.

<!-- DIAGRAM: row-enrichment-fanout -->
[View diagram](../../plans/extra_08_data_engineer_usecase/diagrams/row-enrichment-fanout.mmd)

In [ ]:
# Strict JSON schema for ticket classification.
# strict=True means the API will refuse rather than emit malformed JSON.
# additionalProperties=False prevents the model from adding stray keys.
# enum on intent and urgency keeps the column values stable across runs.
CLASSIFICATION_SCHEMA = {
    "name": "ticket_classification",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "intent": {
                "type": "string",
                "enum": [
                    "account_inquiry",
                    "card_services",
                    "loans",
                    "investments",
                    "complaint",
                    "general",
                ],
            },
            "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
            "product_mentioned": {"type": "string"},
        },
        "required": ["intent", "urgency", "product_mentioned"],
    },
}

# System prompt is short and deterministic - we are not generating prose, we are filling fields.
CLASSIFY_SYSTEM = (
    "You classify Barclays customer support tickets. "
    "Return only the schema fields. "
    "If no specific product is mentioned, set product_mentioned to 'none'."
)


def classify_with_schema(body: str) -> Dict[str, str]:
    """Run a single ticket through Structured Outputs classification."""
    # temperature=0 makes the classification reproducible across re-runs - critical for ETL.
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_schema", "json_schema": CLASSIFICATION_SCHEMA},
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM},
            {"role": "user", "content": body},
        ],
    )
    # Structured Outputs guarantees the content parses as JSON conforming to our schema.
    return json.loads(resp.choices[0].message.content)


# Smoke test the function on two of the synthetic templates.
print(classify_with_schema("My card was lost on holiday. Please freeze it immediately."))
print(classify_with_schema("What is the APR on the Personal Loan?"))

## 🛡️ Enrichment 2: PII redaction before logging or model calls

In a chatbot the PII guardrail is the FIRST thing that runs against user input. Same here, with one change in framing: in a batch pipeline the **log** is the bigger risk. A pipeline that runs nightly can quietly write 50,000 raw NI numbers to CloudWatch in a year. That is a reportable GDPR Article 5(1)(c) breach.

So the rule for this pipeline is: redact first, log only the redacted body, never the raw body. The redaction registry is the same one from Topic 8.

In [ ]:
# Frozen dataclass so a guardrail result cannot be mutated downstream.
# field(default_factory=dict) is the supported pattern for mutable defaults in frozen dataclasses.
@dataclass(frozen=True)
class GuardrailResult:
    passed: bool
    reason: str
    redacted_text: Optional[str] = None
    details: Dict[str, Any] = field(default_factory=dict)


# Same UK PII registry as Topic 8.
# NI number alphabet rules follow HMRC NIM39110 (excluded letters in prefix positions 1 and 2).
# Sort code: 6 digits with optional separators.
# Account number: exactly 8 digits with negative lookarounds so we do not match a longer number.
# Postcode: full UK pattern including the GIR 0AA reserved code.
PII_PATTERNS: Dict[str, re.Pattern] = {
    "uk_ni_number": re.compile(
        r"\b[A-CEGHJ-PR-TW-Z][A-CEGHJ-NPR-TW-Z]\s?\d{2}\s?\d{2}\s?\d{2}\s?[A-D]\b",
        re.IGNORECASE,
    ),
    "uk_sort_code": re.compile(r"\b\d{2}[-\s]?\d{2}[-\s]?\d{2}\b"),
    "uk_account_number": re.compile(r"(?<!\d)\d{8}(?!\d)"),
    "uk_postcode": re.compile(
        r"\b([Gg][Ii][Rr] 0[Aa]{2}|"
        r"[A-Za-z][0-9][A-Za-z0-9]?\s?[0-9][A-Za-z]{2}|"
        r"[A-Za-z][A-Za-z][0-9][A-Za-z0-9]?\s?[0-9][A-Za-z]{2})\b"
    ),
}


def detect_pii(text: str) -> GuardrailResult:
    """Redact UK PII in a single string. Returns a GuardrailResult with redacted_text."""
    # Defensive: empty or non-string input is treated as a pass with empty redaction.
    if not isinstance(text, str) or not text:
        return GuardrailResult(passed=True, reason="empty", redacted_text=text or "")
    # NFKC normalisation defends against full-width digits and homoglyph evasion.
    norm = unicodedata.normalize("NFKC", text)
    redacted = norm
    matched: Dict[str, int] = {}
    for label, pattern in PII_PATTERNS.items():
        hits = pattern.findall(redacted)
        if hits:
            matched[label] = len(hits)
            redacted = pattern.sub(f"[REDACTED_{label.upper()}]", redacted)
    if matched:
        return GuardrailResult(
            passed=False,
            reason="pii_detected",
            redacted_text=redacted,
            details={"matched": matched},
        )
    return GuardrailResult(passed=True, reason="no_pii", redacted_text=redacted)


# Smoke test on a synthetic input that contains all four PII categories.
print(detect_pii("My account is 12345678 and sort code 12-34-56. NI: AB123456C. I live in SW1A 1AA."))

## 📚 Enrichment 3: RAG-grounded suggested response

The classification step gave us the intent. Now we want a draft reply grounded in the Barclays product PDFs we loaded in Section 0. This is exactly the Topic 6 RAG pattern, applied per row.

A note on web search: in the chatbot, Topic 7 added live web search for queries about current rates. In a **batch** pipeline running 50,000 rows nightly, web search is usually the wrong call. It is slow (1-3 seconds per call), it costs more, and it provides freshness you do not need on a 24-hour pipeline. If a row genuinely needs live data, route it to a separate, smaller "live-data" sub-pipeline. The bulk pipeline stays on the static vector store.

In [ ]:
# Retrieve the top-N matching chunks from the ChromaDB collection we built in Section 0.
def retrieve(query: str, n_results: int = 3) -> List[str]:
    """Embed the query and return the top-n matching chunk strings."""
    q_emb = embed_texts([query])[0]
    results = collection.query(query_embeddings=[q_emb], n_results=n_results)
    # results["documents"] is a list of lists (one per query vector); we sent one query so [0].
    docs = results.get("documents", [[]])
    return docs[0] if docs else []


# Grounded prompt: hard rule that the model answers ONLY from the provided context.
# This is the Topic 6 system-message pattern: the model never supplements with training-data guesses.
GROUND_SYSTEM = (
    "You are a Barclays product knowledge assistant. "
    "Answer the customer's question using ONLY the provided context. "
    "If the answer is not in the context, say 'I do not have that information.' "
    "Do not give personal recommendations; share factual product information only."
)


def rag_ground(redacted_body: str) -> Dict[str, Any]:
    """Run a single redacted ticket through the RAG pipeline and return the answer + token usage."""
    # Pull top-3 chunks. Fewer = less context budget, more = more cost per row.
    chunks_used = retrieve(redacted_body, n_results=3)
    context = "\n\n---\n\n".join(chunks_used) if chunks_used else "(no context available)"
    user_block = f"Context:\n{context}\n\nCustomer message:\n{redacted_body}"

    # temperature=0.2 - a touch of variation in phrasing is fine; classification stayed at 0.
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": GROUND_SYSTEM},
            {"role": "user", "content": user_block},
        ],
    )
    answer = resp.choices[0].message.content
    # Capture token usage so the cost report can sum these across all rows.
    return {
        "answer": answer,
        "n_chunks_used": len(chunks_used),
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
    }


# Smoke test the function end-to-end on a single ticket body.
demo = rag_ground("What is the APR on the Personal Loan?")
print(demo["answer"][:300])
print("chunks_used=", demo["n_chunks_used"], "tokens=", demo["prompt_tokens"], "/", demo["completion_tokens"])

## ✅ Enrichment 4: output validation

The model can still produce a personal recommendation by accident, even when grounded. The Topic 8 output validator catches it: regex pass for "you should", "I recommend", "best for you", then OpenAI Moderation API for harmful content. We mark the row `output_safe=False` if validation fails. We do NOT discard the row; we keep it and let downstream review pick it up. **Discarding rows silently is the worst outcome in a data pipeline.**

In [ ]:
# Same recommendation regex registry as Topic 8.
# These phrases are the giveaway tells of a personal recommendation - the FCA boundary case.
RECOMMENDATION_PATTERNS: List[re.Pattern] = [
    re.compile(r"\byou should (open|take|invest|buy|switch|move|consider)\b", re.IGNORECASE),
    re.compile(r"\bi recommend (that you )?\b", re.IGNORECASE),
    re.compile(r"\b(is|are) best for (you|people like you|your situation)\b", re.IGNORECASE),
]


def validate_output(text: str) -> GuardrailResult:
    """Two-layer output validator: regex first, then OpenAI Moderation API."""
    # Defensive: empty text passes (nothing to validate).
    if not isinstance(text, str) or not text:
        return GuardrailResult(passed=True, reason="empty")

    # Layer 1 - cheap regex pass for personal-recommendation phrasing.
    for pattern in RECOMMENDATION_PATTERNS:
        if pattern.search(text):
            return GuardrailResult(
                passed=False,
                reason="recommendation_phrase",
                details={"pattern": pattern.pattern},
            )

    # Layer 2 - free OpenAI Moderation API for harmful content categories.
    # We fail SAFE on any moderation API error: the row gets flagged for human review.
    try:
        mod = client.moderations.create(model=MODERATION_MODEL, input=text)
        if mod.results[0].flagged:
            return GuardrailResult(
                passed=False,
                reason="moderation_flagged",
                details={"categories": dict(mod.results[0].categories)},
            )
    except OpenAIError as exc:
        log.error("Moderation error, failing safe: %s", exc)
        return GuardrailResult(passed=False, reason="moderation_error_fail_safe")

    return GuardrailResult(passed=True, reason="ok")


# Smoke test - factual passes, personal recommendation gets blocked.
print(validate_output("The current Personal Loan representative APR is 6.5 percent."))
print(validate_output("You should take the Personal Loan because it is best for you."))

## 🔧 Composition: enrich_row and enrich_dataframe

Now we wire the four enrichments into one `enrich_row(row)` callable, wrap every LLM call in a tenacity retry, and log a single JSON line per row. Then we map it across the DataFrame.

The retry policy is the Topic 9 policy: `wait_random_exponential(multiplier=1, max=60)` with `stop_after_attempt(6)` for transient OpenAI errors. We do NOT retry on schema validation errors or PII regex errors - those are deterministic and would just fail again.

We catch any exception at the row level and mark `enrichment_status="failed"`. This is the **dead-letter pattern**: a single bad row never crashes a 50,000-row job.

In [ ]:
# Only retry on transient OpenAI errors. Deterministic errors (schema, PII regex) would just fail again.
TRANSIENT = (RateLimitError, APIConnectionError, APITimeoutError)


@retry(
    stop=stop_after_attempt(6),
    wait=wait_random_exponential(multiplier=1, max=60),
    retry=retry_if_exception_type(TRANSIENT),
    reraise=True,
)
def safe_classify(body: str) -> Dict[str, str]:
    """Classification call wrapped in tenacity retry."""
    return classify_with_schema(body)


@retry(
    stop=stop_after_attempt(6),
    wait=wait_random_exponential(multiplier=1, max=60),
    retry=retry_if_exception_type(TRANSIENT),
    reraise=True,
)
def safe_ground(redacted_body: str) -> Dict[str, Any]:
    """RAG ground call wrapped in tenacity retry."""
    return rag_ground(redacted_body)


def enrich_row(row: Dict[str, Any]) -> Dict[str, Any]:
    """Run a single ticket through all four enrichments. Never raises."""
    started = time.perf_counter()
    out: Dict[str, Any] = dict(row)
    try:
        # Step 1 - PII redaction FIRST so the body that goes to the LLM and the log is clean.
        pii = detect_pii(row["body"])
        out["body_redacted"] = pii.redacted_text
        out["pii_matched"] = pii.details.get("matched", {})

        # Step 2 - Structured-output classification on the redacted body.
        cls = safe_classify(out["body_redacted"])
        out.update(cls)

        # Step 3 - RAG-grounded suggested response, also from the redacted body.
        ground = safe_ground(out["body_redacted"])
        out["suggested_response"] = ground["answer"]
        out["rag_chunks_used"] = ground["n_chunks_used"]
        out["prompt_tokens"] = ground["prompt_tokens"]
        out["completion_tokens"] = ground["completion_tokens"]

        # Step 4 - Output validation against the FCA personal-recommendation boundary.
        check = validate_output(out["suggested_response"])
        out["output_safe"] = check.passed
        out["output_reason"] = check.reason

        out["enrichment_status"] = "ok"
    except Exception as exc:
        # Dead-letter pattern: never let one bad row crash the whole DataFrame run.
        out["enrichment_status"] = "failed"
        out["enrichment_error"] = str(exc)[:200]
        out.setdefault("body_redacted", "")
        out.setdefault("prompt_tokens", 0)
        out.setdefault("completion_tokens", 0)
    finally:
        out["latency_ms"] = int((time.perf_counter() - started) * 1000)
        # Log only structural fields - never the raw body.
        log.info(json.dumps({
            "ticket_id": out.get("ticket_id"),
            "status": out.get("enrichment_status"),
            "intent": out.get("intent"),
            "output_safe": out.get("output_safe"),
            "latency_ms": out.get("latency_ms"),
            "prompt_tokens": out.get("prompt_tokens", 0),
            "completion_tokens": out.get("completion_tokens", 0),
        }))
    return out

In [ ]:
# Map enrich_row across the DataFrame. For 50 rows synchronously this takes ~30-90 seconds.
# In production this is where you would swap in a thread pool, an Airflow task, or a Beam DoFn.
def enrich_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply enrich_row to every row of a DataFrame. Returns a new enriched DataFrame."""
    enriched_rows = [enrich_row(r) for r in df.to_dict("records")]
    return pd.DataFrame(enriched_rows)


# Run on a 10-row slice for the demo - 50 rows would take ~1-2 minutes and cost ~3-5 cents.
run_started = time.perf_counter()
enriched_df = enrich_dataframe(tickets_df.head(10))
run_seconds = time.perf_counter() - run_started

log.info("Processed %d rows in %.1f s", len(enriched_df), run_seconds)

# Show the enriched columns - the original `body` is intentionally NOT shown (raw PII discipline).
enriched_df[[
    "ticket_id",
    "intent",
    "urgency",
    "product_mentioned",
    "output_safe",
    "enrichment_status",
    "latency_ms",
]].head()

## 💰 Cost and latency report

Every row carries `prompt_tokens` and `completion_tokens` from the RAG step (the classification step has small fixed cost we are folding into the same accounting for simplicity in this report). We can compute cost per row, total cost, p50 latency, p95 latency, and the failure rate. These five numbers are exactly what your platform team will ask for before this pipeline goes to production.

In [ ]:
# Five-line summary of cost and latency for the run.
def cost_report(df: pd.DataFrame) -> Dict[str, Any]:
    """Aggregate the per-row token usage and latency into a single dict for the platform team."""
    completed = df[df["enrichment_status"] == "ok"]
    total_in = completed["prompt_tokens"].sum()
    total_out = completed["completion_tokens"].sum()
    # Cost = (input tokens / 1000) * input price + (output tokens / 1000) * output price.
    total_cost = (
        (total_in / 1000.0) * GPT4O_INPUT_PRICE_PER_1K
        + (total_out / 1000.0) * GPT4O_OUTPUT_PRICE_PER_1K
    )
    # Latency percentiles - p50 (typical) and p95 (tail) are the two numbers SREs care about.
    latencies = sorted(df["latency_ms"].tolist())
    p50 = latencies[len(latencies) // 2] if latencies else 0
    p95 = latencies[int(len(latencies) * 0.95)] if latencies else 0
    return {
        "rows_total": len(df),
        "rows_ok": int((df["enrichment_status"] == "ok").sum()),
        "rows_failed": int((df["enrichment_status"] == "failed").sum()),
        "total_input_tokens": int(total_in),
        "total_output_tokens": int(total_out),
        "total_cost_usd": round(total_cost, 6),
        "cost_per_row_usd": round(total_cost / max(1, len(completed)), 6),
        "latency_p50_ms": p50,
        "latency_p95_ms": p95,
    }


# Print the report as JSON so the same object can be shipped to CloudWatch / Datadog directly.
print(json.dumps(cost_report(enriched_df), indent=2))

## 🔀 When to use the OpenAI Batch API instead

The synchronous pipeline above is the right default up to about 1,000 rows per run. Beyond that, two options exist:

1. Threaded parallel sync calls (e.g. `concurrent.futures.ThreadPoolExecutor` with 8-32 workers) for sub-hour SLAs.
2. The OpenAI **Batch API** for jobs where a 24-hour SLA is acceptable. The Batch API runs your requests asynchronously over up to 24 hours and returns results at a 50 percent discount on per-token pricing.

Rule of thumb:

- Under 1k rows/day or interactive: stay synchronous.
- 1k-100k rows/day with a sub-hour SLA: parallel synchronous with a thread pool, plus tenacity. Watch your tier rate limits.
- Over 100k rows/day or any non-urgent SLA (overnight backfills, monthly re-classifications, historical re-scoring): use the Batch API.

<!-- DIAGRAM: sync-vs-batch -->
[View diagram](../../plans/extra_08_data_engineer_usecase/diagrams/sync-vs-batch.mmd)

The cell below sketches the JSONL format the Batch API expects. We do NOT submit it (a real batch can take 24 hours). Treat it as the file-format reference you would use in your own job runner.

In [ ]:
# Build a JSONL file in the Batch API format. We do NOT call client.batches.create here -
# a real batch run blocks for up to 24 hours and is out of scope for this demo.
def build_batch_jsonl(df: pd.DataFrame, path: str) -> str:
    """Write one JSON line per row in the OpenAI Batch API request format."""
    with open(path, "w") as fh:
        for _, row in df.iterrows():
            # Each line is a complete batch request: custom_id (your own correlation id),
            # method, url (the endpoint to hit), and body (the same payload you would
            # pass to client.chat.completions.create directly).
            request = {
                "custom_id": row["ticket_id"],
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "temperature": 0,
                    "response_format": {"type": "json_schema", "json_schema": CLASSIFICATION_SCHEMA},
                    "messages": [
                        {"role": "system", "content": CLASSIFY_SYSTEM},
                        {"role": "user", "content": row["body"]},
                    ],
                },
            }
            fh.write(json.dumps(request) + "\n")
    return path


# Write the batch file to /tmp and confirm. The next step in a real pipeline would be
# client.files.create + client.batches.create, then poll for completion.
batch_path = build_batch_jsonl(tickets_df, "/tmp/batch_classify.jsonl")
log.info("Wrote batch JSONL to %s (%d rows). Submit step is intentionally NOT run.", batch_path, len(tickets_df))

## 📋 Production checklist for data engineers

Before you ship a pipeline like this to a real ETL schedule, walk down this list:

1. **Idempotency**: every row carries a stable `ticket_id`. If the job retries, the downstream merge should be UPSERT on `ticket_id`, not INSERT. Otherwise a transient failure produces duplicate enriched rows.
2. **Retry storms**: tenacity backoff is per-call. If the OpenAI API is fully down, a 50,000-row job with 6 retries each becomes 300,000 calls hammering a degraded service. Add a circuit breaker (e.g. fail the whole job if the first 50 rows fail) before turning on a large schedule.
3. **Dead-letter queue**: rows with `enrichment_status="failed"` should land in a separate table or S3 prefix for manual inspection. Do not silently drop them; do not re-queue them in the same run.
4. **Schema evolution**: the Structured Outputs `CLASSIFICATION_SCHEMA` is checked into source control. When you add a new intent (e.g. `mortgage`), bump a schema version field; do not break downstream consumers.
5. **Monitoring**: emit four metrics per run - `rows_total`, `rows_ok`, `rows_failed`, `total_cost_usd`. Alarm on `rows_failed / rows_total > 5 percent` and on `total_cost_usd` exceeding budget.
6. **PII in logs**: log only redacted bodies and only structural fields (intent, urgency, status). Never log raw `body`. CloudWatch retention is typically months; raw PII in logs is the easiest way to a GDPR incident.
7. **Cost ceiling**: keep `EMBED_PRICE_PER_1K` and the gpt-4o pricing constants in one place. When OpenAI changes prices, you change one file. Consider a hard pre-flight estimate that aborts the run if the projected cost exceeds a budget envelope.
8. **Model versioning**: pin to a specific model snapshot (e.g. `gpt-4o-2024-08-06`) for any production schedule so an upstream model change does not silently shift your enrichment distribution. The `gpt-4o` alias is fine for development; pin for production.
9. **Re-runs and backfills**: a backfill across 6 months of historical tickets should reuse the same `enrich_row` function. The function must be deterministic enough (`temperature=0` on classification) that a re-run produces equivalent results. Keep classification temperature at 0; the grounded answer can be slightly higher.
10. **Local dry runs**: every change to the pipeline should run on a 10-row sample before being scheduled. The cost report cell above is exactly that pre-flight.

## 🎁 Wrap

You have seen the same primitives from the chatbot - Structured Outputs, RAG, PII redaction, output validation, retries, cost tracking - composed into a row-level batch enrichment pipeline. The chatbot and the pipeline share 90 percent of the code; only the **shape** differs (request/response loop vs row fan-out). When the next request comes in for "we need an LLM somewhere in this ETL", you already have the building blocks.